# Class 59 — DeepEval with Groq

This copy uses Groq for both the RAG answer model and DeepEval's LLM-as-a-judge metrics, so it does not require an OpenAI API key. The embedding model runs locally through Hugging Face.

Set `GROQ_API_KEY` in a `.env` file in the workspace root or in your environment before running the notebook. If needed, install the notebook dependencies with:

```python
%pip install -U deepeval langchain-groq langchain-huggingface sentence-transformers python-dotenv
```

Groq recommends `openai/gpt-oss-20b` as the replacement for the retired `llama-3.1-8b-instant` model on free/developer-tier accounts. The `openai/` prefix is part of Groq’s model ID; requests still go to Groq and do not require an OpenAI key. Check your Groq plan for current quotas and pricing.

DeepEval defaults to OpenAI unless each LLM-based metric is given a custom model. The Groq judge below is passed to the sample metrics at the end of the notebook. See [Groq model deprecations](https://console.groq.com/docs/deprecations) and [supported models](https://console.groq.com/docs/models).

The RAG demo uses synchronous LangChain calls for compatibility with the Python 3.14 notebook kernel.

In [69]:
print("all ok")

all ok


https://github.com/confident-ai/deepeval

https://deepeval.com/

https://www.confident-ai.com/

In [70]:
#RAG Evaluation
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
)

In [71]:
# Agent Evaluation
from deepeval.metrics import (
    # Reasoning / Planning
    PlanQualityMetric,
    PlanAdherenceMetric,

    # Tool / Action
    ToolCorrectnessMetric,
    ArgumentCorrectnessMetric,

    # Overall Execution
    TaskCompletionMetric,
    StepEfficiencyMetric,
)

from deepeval.metrics import (
    ToolUseMetric,
    GoalAccuracyMetric,
)

from deepeval.metrics import (
    ConversationCompletenessMetric,
    TurnRelevancyMetric,
    RoleAdherenceMetric,
    KnowledgeRetentionMetric,
)

In [72]:
# Safety Evaluation
from deepeval.metrics import (
    BiasMetric,
    ToxicityMetric,
    NonAdviceMetric,
    MisuseMetric,
    PIILeakageMetric,
    RoleViolationMetric,
)

In [73]:
# General LLM / Output Evaluation
from deepeval.metrics import (
    HallucinationMetric,
    PromptAlignmentMetric,
    JsonCorrectnessMetric,
    SummarizationMetric,
    ExactMatchMetric,
)

In [74]:
# Custom LLM-as-a-Judge
from deepeval.metrics import (
    GEval,
    DAGMetric,
    ArenaGEval,
)

In [75]:
from deepeval.test_case import (
    LLMTestCase,
    ToolCall,
    SingleTurnParams,
)
from deepeval.test_case import (
    ConversationalTestCase,
    Turn,
)
from deepeval.test_case import (
    ArenaTestCase,
    Contestant,
)
from deepeval.dataset import (
    EvaluationDataset,
    Golden,
)
from deepeval import (
    evaluate,
    assert_test,
)
from deepeval.tracing import (
    observe,
    update_current_span,
)

In [76]:
# from deepeval import evaluate, assert_test

# from deepeval.dataset import (
#     EvaluationDataset,
#     Golden,
# )

# from deepeval.test_case import (
#     LLMTestCase,
#     ConversationalTestCase,
#     Turn,
#     ToolCall,
#     SingleTurnParams,
#     ArenaTestCase,
#     Contestant,
# )

# from deepeval.metrics import (
#     # RAG
#     AnswerRelevancyMetric,
#     FaithfulnessMetric,
#     ContextualRelevancyMetric,
#     ContextualPrecisionMetric,
#     ContextualRecallMetric,

#     # Agent
#     PlanQualityMetric,
#     PlanAdherenceMetric,
#     ToolCorrectnessMetric,
#     ArgumentCorrectnessMetric,
#     TaskCompletionMetric,
#     StepEfficiencyMetric,
#     ToolUseMetric,
#     GoalAccuracyMetric,

#     # Chatbot
#     ConversationCompletenessMetric,
#     TurnRelevancyMetric,
#     RoleAdherenceMetric,
#     KnowledgeRetentionMetric,

#     # Safety
#     BiasMetric,
#     ToxicityMetric,
#     NonAdviceMetric,
#     MisuseMetric,
#     PIILeakageMetric,
#     RoleViolationMetric,

#     # General
#     HallucinationMetric,
#     PromptAlignmentMetric,
#     JsonCorrectnessMetric,
#     SummarizationMetric,
#     ExactMatchMetric,

#     # Custom
#     GEval,
#     DAGMetric,
#     ArenaGEval,
# )

# from deepeval.tracing import (
#     observe,
#     update_current_span,
# )

In [77]:
import os
import asyncio
from pathlib import Path

import pandas as pd

from dotenv import load_dotenv
from tqdm.auto import tqdm

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

In [78]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_MODEL = "openai/gpt-oss-20b"

if not os.getenv("GROQ_API_KEY"):
    raise ValueError(
        "GROQ_API_KEY is missing. Add it to the .env file or environment."
    )

In [79]:
from pydantic import BaseModel
from deepeval.models import DeepEvalBaseLLM


class GroqJudge(DeepEvalBaseLLM):
    def __init__(self, model_name: str = GROQ_MODEL):
        self._model_name = model_name
        super().__init__(model=model_name)

    def load_model(self) -> ChatGroq:
        return ChatGroq(model=self._model_name, temperature=0)

    def generate(
        self, prompt: str, schema: type[BaseModel] | None = None
    ) -> str | BaseModel:
        if schema is not None:
            return self.model.with_structured_output(schema, method="json_schema").invoke(prompt)

        content = self.model.invoke(prompt).content
        if not isinstance(content, str):
            raise TypeError("Groq returned non-text content for a text prompt.")
        return content

    async def a_generate(
        self, prompt: str, schema: type[BaseModel] | None = None
    ) -> str | BaseModel:
        return await asyncio.to_thread(self.generate, prompt, schema)

    def get_model_name(self) -> str:
        return self._model_name


groq_judge = GroqJudge()

In [80]:
documents = [
    Document(
        page_content="""
Ragas is an open-source evaluation framework designed
for evaluating and improving LLM applications.

It supports RAG evaluation, agent evaluation,
custom metrics, datasets, experiments,
and synthetic test data generation.
""".strip(),
        metadata={"id": "ragas_intro"},
    ),

    Document(
        page_content="""
Ragas experiments allow developers to systematically
evaluate AI applications.

Experiments can track evaluation results,
compare different application versions,
and store experiment outputs.
""".strip(),
        metadata={"id": "ragas_experiments"},
    ),

    Document(
        page_content="""
Ragas metrics measure different properties
of AI systems.

Examples include factual correctness,
faithfulness, answer relevancy,
retrieval quality, and agent behavior.
""".strip(),
        metadata={"id": "ragas_metrics"},
    ),

    Document(
        page_content="""
RAG stands for Retrieval-Augmented Generation.

A RAG system retrieves relevant information
from a knowledge source and provides that
information to an LLM before generating
the final answer.
""".strip(),
        metadata={"id": "rag_intro"},
    ),
]

In [81]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7581.71it/s]


In [82]:
vector_store = InMemoryVectorStore(
    embedding=embedding_model,
)

vector_store.add_documents(documents)

['88f899ab-0f81-4e59-a00c-1d2ec172da3c',
 'cd843d1b-2914-4ea8-9c10-2d6f17929b9a',
 '64f3be9f-71b1-4337-a8c5-d1e6b6a41cd8',
 '32490266-f2fc-46d2-a6bb-8ae1e128a246']

In [83]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 2},
)

In [84]:
llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0,
)

In [85]:
rag_prompt = ChatPromptTemplate.from_template(
    """
You are a helpful AI assistant.

Answer the user's question using the provided context.

If the context does not contain enough information,
say that you do not have enough information.

CONTEXT:

{context}


QUESTION:

{question}


ANSWER:
"""
)

In [86]:
rag_chain = (
    rag_prompt
    | llm
    | StrOutputParser()
)

In [87]:
def rag_query(question):

    retrieved_documents = retriever.invoke(question)

    retrieved_contexts = [
        document.page_content
        for document in retrieved_documents
    ]

    retrieved_document_ids = [
        document.metadata["id"]
        for document in retrieved_documents
    ]

    context = "\n\n".join(retrieved_contexts)

    answer = rag_chain.invoke(
        {
            "context": context,
            "question": question,
        }
    )

    return {
        "answer": answer,
        "retrieved_contexts": retrieved_contexts,
        "retrieved_document_ids": retrieved_document_ids,
    }

In [88]:
question = "What is RAG and how does it work?"

In [89]:
rag_query(question)

{'answer': '**RAG (Retrieval‑Augmented Generation)** is a technique that combines two key steps to produce more accurate and context‑rich responses from a large language model (LLM):\n\n1. **Retrieval** – The system first queries a knowledge source (such as a database, document collection, or web index) to fetch the most relevant pieces of information for the user’s query.  \n2. **Augmented Generation** – The retrieved snippets are then fed into the LLM as part of its prompt. The model uses this additional context to generate the final answer, grounding its response in up‑to‑date or domain‑specific facts that may not be fully encoded in its training data.\n\nIn short, RAG augments the LLM’s generative capabilities with real‑time, external knowledge, enabling it to answer questions more accurately and reliably.',
 'retrieved_contexts': ['RAG stands for Retrieval-Augmented Generation.\n\nA RAG system retrieves relevant information\nfrom a knowledge source and provides that\ninformation t

## Evaluate the RAG app with Groq-backed DeepEval metrics

In [90]:
evaluation_output = rag_query(question)

test_case = LLMTestCase(
    input=question,
    actual_output=evaluation_output["answer"],
    retrieval_context=evaluation_output["retrieved_contexts"],
)

metrics = [
    AnswerRelevancyMetric(model=groq_judge),
    FaithfulnessMetric(model=groq_judge),
]

for metric in metrics:
    metric.measure(test_case)
    print(f"{metric.__class__.__name__}: {metric.score:.2f} — {metric.reason}")

/workspaces/Full-Stack-GenAI-Bootcamp-1.0/.venv/lib/python3.14/site-packages/rich/live.py:260: UserWarning: install
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/workspaces/Full-Stack-GenAI-Bootcamp-1.0/.venv/lib/python3.14/site-packages/rich/live.py:260: UserWarning: install
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/workspaces/Full-Stack-GenAI-Bootcamp-1.0/.venv/lib/python3.14/site-packages/rich/live.py:260: UserWarning: install
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/workspaces/Full-Stack-GenAI-Bootcamp-1.0/.venv/lib/python3.14/site-packages/rich/live.py:260: UserWarning: install
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/workspaces/Full-Stack-GenAI-Bootcamp-1.0/.venv/lib/python3.14/site-packages/rich/live.py:260: UserWarning: install
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

AnswerRelevancyMetric: 1.00 — The score is 1.00 because the answer fully addressed the question with no irrelevant content.


/workspaces/Full-Stack-GenAI-Bootcamp-1.0/.venv/lib/python3.14/site-packages/rich/live.py:260: UserWarning: install
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/workspaces/Full-Stack-GenAI-Bootcamp-1.0/.venv/lib/python3.14/site-packages/rich/live.py:260: UserWarning: install
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

FaithfulnessMetric: 1.00 — The score is 1.00 because there are no contradictions, indicating perfect faithfulness.
